# 01. 강화학습 환경 다루기 — Gymnasium

강화학습은 **에이전트(agent)** 가 **환경(environment)** 과 상호작용하며,
보상을 최대화하는 행동 정책을 학습하는 분야입니다.

이 노트북에서는 강화학습의 표준 도구 **Gymnasium** 으로 환경의 기본 개념을 익힙니다.
대표 예제인 **CartPole**(막대를 쓰러뜨리지 않게 카트를 좌우로 움직이는 문제)을 사용합니다.

1. 환경 구조 (관측·행동·보상)
2. 환경과 한 스텝씩 상호작용
3. 무작위 정책의 성능 측정 (학습 전 기준선)

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

print('gymnasium', gym.__version__)

## 1. 환경 구조

CartPole 환경을 만들고 관측 공간(observation space)과 행동 공간(action space)을 확인합니다.

- **관측**: 카트 위치·속도, 막대 각도·각속도 (4개 실수)
- **행동**: 왼쪽(0) 또는 오른쪽(1) — 2개의 이산 행동
- **보상**: 막대가 서 있는 매 스텝마다 +1

In [ ]:
env = gym.make('CartPole-v1')
print('관측 공간:', env.observation_space)
print('행동 공간:', env.action_space)

obs, info = env.reset(seed=0)
print('초기 관측:', obs)

## 2. 환경과 상호작용

`step(action)` 으로 행동을 취하면 환경은 (다음 관측, 보상, 종료여부, 중단여부, 정보)를 돌려줍니다.
한 에피소드는 막대가 쓰러지거나(terminated) 최대 길이에 도달(truncated)하면 끝납니다.

In [ ]:
obs, info = env.reset(seed=0)
total_reward = 0
for step in range(10):
    action = env.action_space.sample()      # 무작위 행동
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    print(f'스텝 {step}: 행동={action}, 보상={reward}, 종료={terminated}')
    if terminated or truncated:
        break
print('누적 보상:', total_reward)

## 3. 무작위 정책 기준선

학습 전, 무작위로 행동하는 에이전트의 평균 성능을 측정합니다.
이 값이 이후 학습한 에이전트와 비교할 **기준선**이 됩니다. CartPole의 최대 점수는 500입니다.

In [ ]:
def run_episodes(policy_fn, n=20):
    rewards = []
    for _ in range(n):
        obs, _ = env.reset()
        ep_r, done = 0, False
        while not done:
            action = policy_fn(obs)
            obs, r, terminated, truncated, _ = env.step(action)
            ep_r += r
            done = terminated or truncated
        rewards.append(ep_r)
    return np.array(rewards)

random_rewards = run_episodes(lambda obs: env.action_space.sample())
print(f'무작위 정책 평균 보상: {random_rewards.mean():.1f} (최대 500)')
env.close()

### 정리

- Gymnasium 환경의 관측·행동·보상 구조와 `reset`/`step` 상호작용 방식을 익혔습니다.
- 무작위 정책의 평균 보상을 측정해 학습 전 기준선을 잡았습니다 (보통 20~30점 수준).
- 다음 노트북에서는 이 환경을 **실제로 학습**해 훨씬 높은 점수를 얻습니다.